# 03 - Accessibility prototype

Purpose: prototype catchment and accessibility maths before the R5 travel-time matrix exists.

This notebook has two layers: a tiny synthetic sanity check, then a real Stage 0 distance-based proxy using the LSOA origins and candidate stops from Notebook 2.

## Important limitation

This is **not** the final accessibility model. It uses straight-line centroid-to-stop distance converted to walking time at 80 metres/minute. Milestone 1 will replace this with an R5 walk-plus-transit matrix.

For now, this proxy is useful because it validates the catchment maths, the deprivation aggregation, and the baseline-vs-scenario reporting shape.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import os
import subprocess
import sys

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))
import matplotlib.pyplot as plt

DATA_PROCESSED = ROOT / "data" / "processed"

DATA_PROCESSED

### Output: processed data directory

All inputs for this notebook come from `data/processed`, created by Notebook 2 and the Stage 0 scripts.

## Synthetic sanity check

Keep this tiny example in the notebook because it makes the catchment definition easy to audit before the real data adds noise.

In [ ]:
@dataclass(frozen=True)
class Origin:
    origin_id: str
    population: int
    imd_decile: int


toy_origins = [
    Origin("lsoa_a", population=900, imd_decile=1),
    Origin("lsoa_b", population=700, imd_decile=2),
    Origin("lsoa_c", population=500, imd_decile=7),
]

toy_travel_times = {
    ("lsoa_a", "stop_1"): 8,
    ("lsoa_a", "stop_2"): 18,
    ("lsoa_a", "stop_3"): 26,
    ("lsoa_b", "stop_1"): 14,
    ("lsoa_b", "stop_2"): 9,
    ("lsoa_b", "stop_3"): 22,
    ("lsoa_c", "stop_1"): 21,
    ("lsoa_c", "stop_2"): 11,
    ("lsoa_c", "stop_3"): 7,
}

toy_baseline_covered_origins = {"lsoa_c"}

In [ ]:
def covered_origins(
    selected_stops: set[str],
    travel_time_minutes: dict[tuple[str, str], float],
    threshold_minutes: float,
) -> set[str]:
    covered: set[str] = set()
    for origin_id, stop_id in travel_time_minutes:
        if stop_id in selected_stops and travel_time_minutes[(origin_id, stop_id)] <= threshold_minutes:
            covered.add(origin_id)
    return covered


def population_in_deciles(
    origins: list[Origin],
    covered_origin_ids: set[str],
    max_decile: int,
) -> int:
    return sum(
        origin.population
        for origin in origins
        if origin.origin_id in covered_origin_ids and origin.imd_decile <= max_decile
    )


toy_selected = {"stop_1", "stop_2"}
toy_scenario_covered = covered_origins(toy_selected, toy_travel_times, threshold_minutes=15)

{
    "toy_covered_origins": toy_scenario_covered,
    "baseline_most_deprived_population": population_in_deciles(toy_origins, toy_baseline_covered_origins, max_decile=1),
    "scenario_most_deprived_population": population_in_deciles(toy_origins, toy_scenario_covered, max_decile=1),
    "delta": population_in_deciles(toy_origins, toy_scenario_covered, max_decile=1)
    - population_in_deciles(toy_origins, toy_baseline_covered_origins, max_decile=1),
}

### Output: synthetic sanity check

This confirms the basic rule: an origin is covered if at least one selected stop is reachable within the threshold, and equity population is summed only for the chosen deprivation deciles.

## Real Stage 0 proxy outputs

Build the distance proxy if it is missing. The script creates an origin-candidate matrix, origin-level coverage flags, and summary metrics.

In [ ]:
accessibility_files = {
    "matrix": DATA_PROCESSED / "accessibility_proxy_matrix.csv",
    "origin_accessibility": DATA_PROCESSED / "accessibility_proxy_by_origin.geojson",
    "summary": DATA_PROCESSED / "accessibility_proxy_summary.csv",
    "metadata": DATA_PROCESSED / "stage0_accessibility_metadata.json",
}

def file_status(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for key, path in paths.items():
        rows.append(
            {
                "key": key,
                "path": path.relative_to(ROOT).as_posix(),
                "exists": path.exists(),
                "size_mb": round(path.stat().st_size / 1_000_000, 3) if path.exists() else 0,
            }
        )
    return pd.DataFrame(rows)


if not all(path.exists() for path in accessibility_files.values()):
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "build_stage0_accessibility.py")],
        check=True,
        cwd=ROOT,
    )

file_status(accessibility_files)

### Output: accessibility file status

All rows should be `True`. The matrix is the main bridge into Notebook 4, where the optimiser needs origin-candidate coverage information.

## Load real proxy outputs

Baseline proxy: an origin is directly within walking reach of a rail/Metro interchange candidate.

Scenario proxy: an origin is within walking reach of any candidate stop. This approximates feeder-stop reach, not full transit access.

In [ ]:
accessibility_matrix = pd.read_csv(accessibility_files["matrix"])
origin_accessibility = gpd.read_file(accessibility_files["origin_accessibility"])
accessibility_summary = pd.read_csv(accessibility_files["summary"])
accessibility_metadata = json.loads(accessibility_files["metadata"].read_text())

{
    "origin_count": accessibility_metadata["origin_count"],
    "candidate_count": accessibility_metadata["candidate_count"],
    "matrix_rows": accessibility_metadata["matrix_rows"],
    "method": accessibility_metadata["method"],
}

### Output: proxy metadata

The matrix should have `origin_count × candidate_count` rows. The method text is deliberately included so any chart from this notebook carries the correct caveat.

## Headline proxy summary

The final R5 metric will be more sophisticated, but this table has the same reporting shape: baseline, scenario, delta, threshold, and deprivation scope.

In [ ]:
accessibility_summary

### Output: accessibility proxy summary

`baseline_direct_interchange_population` counts residents whose LSOA centroid is within the threshold of a rail/Metro candidate. `scenario_any_candidate_population` counts residents within the threshold of any candidate stop. The delta is the accessibility gap this feeder concept could address, before replacing the proxy with R5.

## Headline Milestone 1-style metric

Use the 10-minute proxy threshold and the most-deprived decile to mirror the Milestone 1 headline shape.

In [ ]:
headline_proxy = accessibility_summary.loc[
    accessibility_summary["threshold_min"].eq(10)
    & accessibility_summary["population_scope"].eq("most_deprived_decile")
].iloc[0]

headline_proxy.to_dict()

### Output: headline proxy metric

This is not the final number for the README. It is the shape of the number: most-deprived-decile residents within reach under baseline versus scenario.

## Origin-level accessibility EDA

Inspect nearest candidate and nearest interchange walking times by deprivation decile.

In [ ]:
origin_time_summary = (
    origin_accessibility.groupby("imd_decile", as_index=False)
    .agg(
        lsoa_count=("lsoa21cd", "count"),
        population_mid_2024=("population_mid_2024", "sum"),
        median_nearest_candidate_min=("nearest_candidate_walk_time_min", "median"),
        median_nearest_interchange_min=("nearest_interchange_walk_time_min", "median"),
        covered_any_10_min=("covered_any_10_min", "sum"),
        covered_interchange_10_min=("covered_interchange_10_min", "sum"),
    )
    .sort_values("imd_decile")
)

origin_time_summary

### Output: origin-level accessibility by decile

A large gap between nearest candidate time and nearest interchange time is exactly the feeder-access problem the project is trying to expose.

## LSOAs with the largest interchange gap

These are the origins that are close to some candidate stop but comparatively far from rail/Metro interchange access.

In [ ]:
gap_columns = [
    "lsoa21cd",
    "boundary_lsoa21nm",
    "imd_decile",
    "population_mid_2024",
    "nearest_candidate_name",
    "nearest_candidate_walk_time_min",
    "nearest_interchange_name",
    "nearest_interchange_walk_time_min",
]

largest_interchange_gaps = origin_accessibility.copy()
largest_interchange_gaps["interchange_gap_min"] = (
    largest_interchange_gaps["nearest_interchange_walk_time_min"]
    - largest_interchange_gaps["nearest_candidate_walk_time_min"]
)

largest_interchange_gaps[gap_columns + ["interchange_gap_min"]].sort_values(
    ["interchange_gap_min", "population_mid_2024"], ascending=[False, False]
).head(12)

### Output: largest interchange gaps

These LSOAs are useful case-study candidates for the portfolio narrative: they are near local stop infrastructure but not near high-capacity interchange access.

## Map: 10-minute direct-interchange gap

Map origins covered by any candidate stop but not by a rail/Metro interchange within 10 minutes.

In [ ]:
map_frame = origin_accessibility.copy()
map_frame["gap_10_min"] = map_frame["covered_any_10_min"] & ~map_frame["covered_interchange_10_min"]

fig, ax = plt.subplots(figsize=(10, 8))
map_frame.plot(
    column="gap_10_min",
    categorical=True,
    cmap="Set1",
    legend=True,
    linewidth=0.35,
    edgecolor="#4b5563",
    alpha=0.86,
    ax=ax,
)
ax.set_title("10-minute proxy gap: near candidate stop, not near rail/Metro interchange", fontsize=12)
ax.set_axis_off()
plt.tight_layout()

### Output: proxy gap map

`True` polygons are the clearest pre-R5 accessibility gap: they can reach some candidate stop within 10 minutes, but cannot directly reach a rail/Metro interchange within 10 minutes.

## Coverage matrix sample for Notebook 4

Notebook 4 can use this matrix to prototype the optimiser before R5. Each row is an origin-candidate pair with distance, proxy walk time, and threshold flags.

In [ ]:
accessibility_matrix.head(12)

### Output: accessibility matrix sample

The key field for Notebook 4 is `within_10_min` or `within_15_min`, which can become the coverage relation `N(i)` in the MILP prototype.

## EDA findings

- The proxy matrix now uses real study-area LSOAs and real NaPTAN-derived candidate stops.
- Most origins are close to some candidate stop, but many are much farther from rail/Metro interchange candidates.
- The 10-minute most-deprived-decile proxy metric has the same shape as the final Milestone 1 metric, but it must be replaced by R5 before being presented as a result.
- The coverage matrix is ready for Notebook 4's optimiser prototype.
- Production extraction target: move stable coverage/aggregation functions into `app/accessibility/` and test them with small fixtures.